# 02. Tiền xử lý dữ liệu từ đầu (Data Preprocessing from Scratch)

## 1. Mục tiêu và Nguyên lý chống Data Leakage
Trong quy trình học máy chuẩn mực, **Data Leakage (Rò rỉ dữ liệu)** xảy ra khi thông tin từ tập Test vô tình bị đưa vào quá trình huấn luyện mô hình. Ví dụ:
- Tính `mean` hoặc `median` trên toàn bộ tập dữ liệu trước khi chia train/test.
- Học danh sách categories cho One-Hot Encoding trên toàn bộ tập dữ liệu.

Để ngăn chặn triệt để, class `HeartDiseasePreprocessor` được thiết kế theo đúng nguyên tắc:
1. `fit()` chỉ được gọi trên tập **Training Data** để học các thống kê.
2. `transform()` áp dụng các thống kê đã học lên tập **Train** và **Test** một cách độc lập.

In [1]:
import sys
sys.path.append("..")
import numpy as np
import pandas as pd
from src.data_utils import train_test_split
from src.preprocessing import HeartDiseasePreprocessor

# 1. Đọc dữ liệu gốc
df = pd.read_csv("../data/heart_cleveland_upload.csv")
X = df.drop("condition", axis=1)
y = df["condition"]

print(f"Tổng số mẫu dữ liệu: {len(df)}")
print(f"Tỷ lệ nhãn condition:\n{y.value_counts(normalize=True)}")

Tổng số mẫu dữ liệu: 297
Tỷ lệ nhãn condition:
condition
0    0.538721
1    0.461279
Name: proportion, dtype: float64


### 2. Phân chia Train/Test phân tầng (Stratified Train-Test Split)
Hàm `train_test_split` tự cài đặt sử dụng `rng = np.random.default_rng(42)` để chia dữ liệu đảm bảo tỷ lệ nhãn giữa Train và Test hoàn toàn tương đồng.

In [2]:
# Chia tập train/test với tỷ lệ 80/20 và stratify
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=True)

print(f"Kích thước X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"Kích thước X_test: {X_test.shape}, y_test: {y_test.shape}")
print(f"Tỷ lệ nhãn y_train:\n{pd.Series(y_train).value_counts(normalize=True)}")
print(f"Tỷ lệ nhãn y_test:\n{pd.Series(y_test).value_counts(normalize=True)}")

Kích thước X_train: (238, 13), y_train: (238,)
Kích thước X_test: (59, 13), y_test: (59,)
Tỷ lệ nhãn y_train:
condition
0    0.537815
1    0.462185
Name: proportion, dtype: float64
Tỷ lệ nhãn y_test:
condition
0    0.542373
1    0.457627
Name: proportion, dtype: float64


### 3. Huấn luyện Preprocessor trên tập Train
Chúng ta áp dụng `HeartDiseasePreprocessor`:
- **Đặc trưng liên tục (Numerical):** `age`, `trestbps`, `chol`, `thalach`, `oldpeak` -> Điền khuyết bằng `median(X_train)` và chuẩn hóa $z = (x - \mu) / \sigma$.
- **Đặc trưng phân loại (Categorical):** `sex`, `cp`, `fbs`, `restecg`, `exang`, `slope`, `ca`, `thal` -> Điền khuyết bằng `mode(X_train)` và mã hóa One-Hot nhị phân.

In [3]:
preprocessor = HeartDiseasePreprocessor()
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("X_train_processed shape:", X_train_processed.shape)
print("X_test_processed shape:", X_test_processed.shape)
print("Kiểu dữ liệu ma trận đầu ra:", X_train_processed.dtype)
print(f"Số lượng đặc trưng sau One-Hot: {len(preprocessor.feature_names_)}")
print("Danh sách tên đặc trưng:", preprocessor.feature_names_)

X_train_processed shape: (238, 28)
X_test_processed shape: (59, 28)
Kiểu dữ liệu ma trận đầu ra: float64
Số lượng đặc trưng sau One-Hot: 28
Danh sách tên đặc trưng: ['age', 'trestbps', 'chol', 'thalach', 'oldpeak', 'sex_0', 'sex_1', 'cp_0', 'cp_1', 'cp_2', 'cp_3', 'fbs_0', 'fbs_1', 'restecg_0', 'restecg_1', 'restecg_2', 'exang_0', 'exang_1', 'slope_0', 'slope_1', 'slope_2', 'ca_0', 'ca_1', 'ca_2', 'ca_3', 'thal_0', 'thal_1', 'thal_2']


### 4. Kiểm tra khả năng xử lý giá trị chưa từng xuất hiện (Unseen Categories)
Nếu tập Test xuất hiện một giá trị thuộc tính mới (chưa có trong tập Train), preprocessor sẽ gán toàn bộ vector one-hot của thuộc tính đó về 0, không gây lỗi runtime.

In [4]:
# Tạo mẫu thử nghiệm chứa category mới chưa từng có (ví dụ cp = 99)
sample_unseen = X_test.iloc[[0]].copy()
sample_unseen.loc[0, 'cp'] = 99
sample_transformed = preprocessor.transform(sample_unseen)
print("Kích thước sau transform:", sample_transformed.shape)
print("Có chứa NaN không?:", np.isnan(sample_transformed).any())

Kích thước sau transform: (1, 28)
Có chứa NaN không?: False


## 5. Kết luận
- `HeartDiseasePreprocessor` tự cài đặt hoàn toàn bằng NumPy/Pandas đã thay thế trọn vẹn `Pipeline`, `ColumnTransformer`, `StandardScaler` và `OneHotEncoder` của scikit-learn.
- Mọi tham số thống kê (median, mean, std, categories) đều được học thuần túy từ tập Train, đảm bảo tính toàn vẹn của dữ liệu và ngăn ngừa rò rỉ thông tin.